# 15. YOLO 이미지 실습

이 노트북은 앞에서 배운 YOLO 후처리 개념을 실제 이미지 추론 흐름으로 연결합니다.

환경에 `ultralytics`와 YOLO 가중치 파일이 준비되어 있으면 실제 모델 추론을 실행합니다. 준비되어 있지 않아도, 같은 결과 형식의 예제 탐지값으로 시각화와 해석 과정을 끝까지 실행할 수 있습니다.

이번 노트북의 목표는 다음과 같습니다.

- 이미지 한 장을 불러오고 탐지 결과를 그리는 기본 흐름을 익힙니다.
- YOLO 라이브러리 결과에서 bbox, confidence, class를 읽습니다.
- confidence threshold를 바꾸며 결과가 어떻게 달라지는지 확인합니다.
- 모델이 없을 때도 탐지 결과 형식과 시각화 방법을 연습합니다.

In [ ]:
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.unicode_minus'] = False

## 15-1. 실습 이미지 준비

먼저 외부 파일 없이도 실행 가능한 간단한 예제 이미지를 만듭니다. 실제 사진으로 실습하려면 `image_path`를 본인의 이미지 경로로 바꾸면 됩니다.

In [ ]:
def make_demo_image(width=640, height=420):
    image = np.full((height, width, 3), 245, dtype=np.uint8)

    # cat 역할의 붉은 사각형
    image[120:280, 90:260] = np.array([220, 80, 80], dtype=np.uint8)
    image[145:175, 125:155] = 245
    image[145:175, 195:225] = 245

    # dog 역할의 푸른 사각형
    image[150:330, 360:540] = np.array([70, 120, 220], dtype=np.uint8)
    image[185:215, 395:425] = 245
    image[185:215, 475:505] = 245

    return image


image_path = None

if image_path is None:
    image = make_demo_image()
else:
    from PIL import Image
    image = np.array(Image.open(image_path).convert('RGB'))

plt.imshow(image)
plt.title('실습 이미지')
plt.axis('off')
plt.show()

print('image shape:', image.shape)

## 15-2. YOLO 모델 사용 가능 여부 확인

`ultralytics`가 설치되어 있고, 로컬에 가중치 파일이 있으면 실제 추론을 수행합니다.

가중치가 없다면 아래 셀은 예제 탐지 결과를 사용합니다. 이 노트북은 개념 학습용이므로 패키지 설치나 모델 다운로드를 자동으로 실행하지 않습니다.

In [ ]:
use_real_model = False
model_path = Path('yolov8n.pt')

try:
    from ultralytics import YOLO
    if model_path.exists():
        model = YOLO(str(model_path))
        use_real_model = True
        print('YOLO model loaded:', model_path)
    else:
        print('ultralytics는 설치되어 있지만 가중치 파일이 없습니다:', model_path)
except ImportError:
    print('ultralytics가 설치되어 있지 않습니다. 예제 탐지 결과로 진행합니다.')

print('use_real_model:', use_real_model)

## 15-3. 이미지 추론 실행

실제 모델이 있으면 `model.predict`를 호출합니다. 없으면 이전 노트북에서 본 것과 같은 `class`, `score`, `box` 형식의 예제 결과를 만듭니다.

In [ ]:
def fallback_detections_for_demo():
    return [
        {'class': 'cat', 'score': 0.91, 'box': (90, 120, 260, 280)},
        {'class': 'dog', 'score': 0.87, 'box': (360, 150, 540, 330)},
        {'class': 'cat', 'score': 0.32, 'box': (105, 135, 248, 270)},
    ]


if use_real_model:
    results = model.predict(source=image, conf=0.25, verbose=False)
    result = results[0]
    detections = []
    names = result.names
    for box in result.boxes:
        xyxy = box.xyxy[0].cpu().numpy().tolist()
        class_id = int(box.cls[0].cpu().item())
        score = float(box.conf[0].cpu().item())
        detections.append({
            'class': names[class_id],
            'score': score,
            'box': tuple(xyxy),
        })
else:
    detections = fallback_detections_for_demo()

for det in detections:
    print(f"{det['class']:<10} score={det['score']:.3f} box={tuple(round(v, 1) for v in det['box'])}")

## 15-4. 탐지 결과 시각화 함수

탐지 결과를 읽을 때는 클래스 이름과 confidence만 보지 말고, 박스가 실제 객체를 잘 감싸는지도 함께 봐야 합니다.

In [ ]:
def draw_image_detections(image, detections, score_threshold=0.25, title=''):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(image)
    ax.set_title(title or f'Detections | score >= {score_threshold}')
    ax.axis('off')

    colors = ['crimson', 'royalblue', 'seagreen', 'darkorange', 'purple']
    class_to_color = {}

    shown = 0
    for det in detections:
        if det['score'] < score_threshold:
            continue
        class_name = det['class']
        if class_name not in class_to_color:
            class_to_color[class_name] = colors[len(class_to_color) % len(colors)]
        color = class_to_color[class_name]

        x1, y1, x2, y2 = det['box']
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5))
        ax.text(
            x1,
            max(12, y1 - 6),
            f"{class_name} {det['score']:.2f}",
            color='white',
            fontsize=10,
            weight='bold',
            bbox={'facecolor': color, 'alpha': 0.85, 'pad': 2, 'edgecolor': 'none'},
        )
        shown += 1

    plt.show()
    print('shown detections:', shown)


draw_image_detections(image, detections, score_threshold=0.25, title='YOLO 이미지 탐지 결과')

## 15-5. Confidence threshold 바꿔 보기

threshold를 바꾸면 탐지 결과가 달라집니다.

- 낮은 threshold: 놓치는 객체는 줄 수 있지만 오탐이 늘 수 있습니다.
- 높은 threshold: 결과는 깔끔하지만 확신이 낮은 실제 객체를 놓칠 수 있습니다.

In [ ]:
for threshold in [0.25, 0.5, 0.8]:
    kept = [det for det in detections if det['score'] >= threshold]
    print(f'threshold={threshold}: {len(kept)}개')
    for det in kept:
        print(f"  {det['class']:<10} {det['score']:.3f}")

## 15-6. 박스 크기와 중심점 확인하기

탐지 결과는 좌표로 다룰 수 있어야 합니다. 후속 작업에서는 박스 중심점, 너비, 높이를 이용해 추적, 거리 추정, ROI crop 등을 수행합니다.

In [ ]:
def xyxy_to_cxcywh(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)


for det in detections:
    cx, cy, w, h = xyxy_to_cxcywh(det['box'])
    print(f"{det['class']:<10} center=({cx:.1f}, {cy:.1f}) size=({w:.1f}, {h:.1f}) score={det['score']:.2f}")

## 15-7. 탐지된 영역 crop하기

YOLO 결과를 활용할 때는 박스 영역을 잘라서 별도 분류기, OCR, 품질 검사 모델에 넘기는 경우도 많습니다.

In [ ]:
high_conf = [det for det in detections if det['score'] >= 0.5]

if high_conf:
    fig, axes = plt.subplots(1, len(high_conf), figsize=(4 * len(high_conf), 3))
    if len(high_conf) == 1:
        axes = [axes]

    for ax, det in zip(axes, high_conf):
        x1, y1, x2, y2 = [int(round(v)) for v in det['box']]
        crop = image[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
        ax.imshow(crop)
        ax.set_title(f"{det['class']} {det['score']:.2f}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('score >= 0.5인 탐지 결과가 없습니다.')

## 15-8. 실제 이미지로 바꿔 실험하기

본인의 이미지로 실험하려면 위쪽의 `image_path`를 다음처럼 바꿉니다.

```python
image_path = 'data/my_image.jpg'
```

실제 YOLO 모델을 사용하려면 `ultralytics` 설치와 가중치 파일이 필요합니다.

```python
pip install ultralytics
```

그리고 `yolov8n.pt` 같은 가중치 파일을 현재 폴더에 두거나 `model_path`를 해당 파일 경로로 바꿉니다.

## 정리

- 실제 YOLO 추론 결과도 결국 `class`, `confidence`, `xyxy box`로 읽으면 됩니다.
- confidence threshold는 결과 개수와 품질을 조절하는 핵심 값입니다.
- bbox 좌표를 이용하면 시각화뿐 아니라 crop, 추적, 후속 모델 연결도 가능합니다.
- 패키지나 가중치가 없어도 탐지 결과의 구조를 먼저 익히면 실제 모델 결과를 읽는 부담이 줄어듭니다.

다음 노트북 `16_YOLO_영상_실습.ipynb`에서는 이미지 한 장이 아니라 여러 프레임에 같은 추론 흐름을 반복 적용하는 방법을 다룹니다.